# Painting → Music: experiments for the TDT12 report

This notebook runs **all experiments** for the report *Music generation based on paintings*
(TDT12, NTNU). It builds three system variants, generates the music and evaluates it automatically:

| Variant | Emotion source | Composer |
|---|---|---|
| A (baseline) | Colour–mood table of Sandve et al. (2016) | Rule-based |
| B | ArtEmis annotations | Rule-based |
| C | ArtEmis annotations | MusicGen (small) |


In [ ]:
SEED = 2026               
N_PER_QUADRANT = 25      
CLIP_SECONDS = 20        
MIN_ANNOTATIONS = 5     
MIN_AGREEMENT = 0.60    
N_DISTRACTOR_SETS = 10      
N_EMOPIA_PER_QUADRANT = 50 
TARGET_LUFS = -23.0        
SAMPLE_RATE = 32000    

WORK = "/content/painting2music"                      
ARTEMIS_CSV = "/content/artemis_dataset_release_v0.csv" 

In [ ]:
#Install software 
import sys, subprocess

def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
    return r.returncode

# FluidSynth renders MIDI to audio with a General MIDI piano soundfont
sh("apt-get -qq update && apt-get -qq install -y fluidsynth fluid-soundfont-gm")
sh(f"{sys.executable} -m pip install -q pretty_midi pyloudnorm librosa soundfile")

# Essentia with TensorFlow support provides the automatic listener
if sh(f'{sys.executable} -m pip install -q "essentia-tensorflow==2.1b6.dev1389"') != 0:
    sh(f"{sys.executable} -m pip install -q --pre essentia-tensorflow")
ok = sh(f'{sys.executable} -c "import essentia.standard as es; print(es.TensorflowPredict2D)"') == 0
print("\nEssentia available:", ok)
assert ok, "Essentia could not be installed. Restart the runtime (Runtime → Restart session) and run all cells again."


In [ ]:
# Imports and shared helpers
import os, sys, re, json, math, time, glob, shutil, subprocess, zipfile, platform, warnings
import numpy as np
import pandas as pd
import soundfile as sf
import pyloudnorm as pyln
import librosa
import pretty_midi
warnings.filterwarnings("ignore")

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files, IN_COLAB = None, False

DIRS = ["images", "midi", "audio/A", "audio/B", "audio/C", "emopia/raw", "emopia/audio",
        "models", "results", "figures"]
for d in DIRS:
    os.makedirs(os.path.join(WORK, d), exist_ok=True)
P = lambda *parts: os.path.join(WORK, *parts)

QUADS = ["Q1", "Q2", "Q3", "Q4"]
Q_NAMES = {"Q1": "positive / high arousal", "Q2": "negative / high arousal",
           "Q3": "negative / low arousal", "Q4": "positive / low arousal"}
Q_CENTRE = {"Q1": (1.0, 1.0), "Q2": (-1.0, 1.0), "Q3": (-1.0, -1.0), "Q4": (1.0, -1.0)}
ARTEMIS_TO_Q = {"amusement": "Q1", "awe": "Q1", "excitement": "Q1",
                "anger": "Q2", "disgust": "Q2", "fear": "Q2",
                "sadness": "Q3", "contentment": "Q4"}         
MOODS = ["happy", "sad", "calm", "excited", "angry", "sick"]
MOOD_TO_Q = {"happy": "Q1", "excited": "Q1", "angry": "Q2", "sick": "Q2", "sad": "Q3", "calm": "Q4"}
Q_TO_MOOD = {"Q1": "happy", "Q2": "angry", "Q3": "sad", "Q4": "calm"}
VARIANTS = ["A", "B", "C"]
VARIANT_NAMES = {"A": "A: colour table + rules", "B": "B: ArtEmis + rules", "C": "C: ArtEmis + MusicGen"}

def quadrant_of(v, a):
    """Quadrant of a (centred) valence/arousal point."""
    if v >= 0:
        return "Q1" if a >= 0 else "Q4"
    return "Q2" if a >= 0 else "Q3"

# results log
RESULTS_JSON = P("results", "results.json")
LOG = json.load(open(RESULTS_JSON)) if os.path.exists(RESULTS_JSON) else {}

def to_builtin(x):
    if isinstance(x, dict):
        return {str(k): to_builtin(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [to_builtin(v) for v in x]
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return None if np.isnan(x) else float(x)
    if isinstance(x, np.ndarray):
        return to_builtin(x.tolist())
    return x

def log(key, value):
    LOG[key] = to_builtin(value)
    json.dump(LOG, open(RESULTS_JSON, "w"), indent=2)

# audio post-processing shared by all composers 
def finish_clip(wav, sr, path):
    """Mono, resample to SAMPLE_RATE, cut/pad to CLIP_SECONDS, 1 s fade-out, normalise loudness."""
    wav = np.asarray(wav, dtype=np.float32)
    if wav.ndim == 2:
        wav = wav.mean(axis=1) if wav.shape[1] <= 2 else wav.mean(axis=0)
    if sr != SAMPLE_RATE:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SAMPLE_RATE)
    n = int(CLIP_SECONDS * SAMPLE_RATE)
    wav = np.pad(wav, (0, max(0, n - len(wav))))[:n]
    fade = int(1.0 * SAMPLE_RATE)
    wav[-fade:] *= np.linspace(1.0, 0.0, fade, dtype=np.float32)
    meter = pyln.Meter(SAMPLE_RATE)
    loud = meter.integrated_loudness(wav)
    if np.isfinite(loud):
        wav = pyln.normalize.loudness(wav, loud, TARGET_LUFS)
    peak = np.abs(wav).max()
    if peak > 0.99:
        wav = wav * (0.99 / peak)
    sf.write(path, wav.astype(np.float32), SAMPLE_RATE, subtype="PCM_16")
    return path

SOUNDFONT = "/usr/share/sounds/sf2/FluidR3_GM.sf2"

def render_midi(midi_path, wav_path):
    """Render a MIDI file to audio with FluidSynth and the FluidR3 GM soundfont."""
    tmp = wav_path + ".raw.wav"
    subprocess.run(["fluidsynth", "-ni", "-g", "0.8", "-r", "44100", "-F", tmp, SOUNDFONT, midi_path],
                   check=True, capture_output=True)
    wav, sr = sf.read(tmp)
    os.remove(tmp)
    return finish_clip(wav, sr, wav_path)

T0 = time.time()
print("Working folder:", WORK)


In [ ]:
# Load the ArtEmis annotations
if not os.path.exists(ARTEMIS_CSV):
    assert IN_COLAB, f"Put the ArtEmis CSV at {ARTEMIS_CSV}"
    print("Please upload artemis_dataset_release_v0.csv (from the ArtEmis download).")
    uploaded = colab_files.upload()
    name = next(iter(uploaded))
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as z:
            member = [m for m in z.namelist() if m.endswith(".csv") and "artemis" in m.lower()][0]
            with z.open(member) as src, open(ARTEMIS_CSV, "wb") as dst:
                shutil.copyfileobj(src, dst)
    else:
        shutil.move(name, ARTEMIS_CSV)

art = pd.read_csv(ARTEMIS_CSV)
need = {"art_style", "painting", "emotion"}
assert need <= set(art.columns), f"Unexpected columns: {list(art.columns)}"
art["emotion"] = art["emotion"].str.strip().str.lower()
art["quadrant"] = art["emotion"].map(ARTEMIS_TO_Q)
print(f"{len(art):,} annotations of {art.groupby(['art_style', 'painting']).ngroups:,} paintings")
print(art["emotion"].value_counts().to_string())

# Candidate pool
valid = art.dropna(subset=["quadrant"])
counts = (valid.groupby(["art_style", "painting", "quadrant"]).size()
          .unstack(fill_value=0).reindex(columns=QUADS, fill_value=0))
n_ann = counts.sum(axis=1)
share = counts.div(n_ann, axis=0)
pool = pd.DataFrame({"n_annotations": n_ann,
                     "quadrant_artemis": share.idxmax(axis=1),
                     "agreement": share.max(axis=1)}).join(share.add_prefix("share_"))
pool = pool[(pool.n_annotations >= MIN_ANNOTATIONS) & (pool.agreement >= MIN_AGREEMENT)].reset_index()

# The painting's point in the valence–arousal plane
pool["painting_v"] = sum(pool[f"share_{q}"] * Q_CENTRE[q][0] for q in QUADS)
pool["painting_a"] = sum(pool[f"share_{q}"] * Q_CENTRE[q][1] for q in QUADS)
print("\nCandidate paintings per quadrant:")
print(pool["quadrant_artemis"].value_counts().reindex(QUADS).to_string())
log("artemis", {"n_annotations": len(art),
                "n_paintings": int(art.groupby(["art_style", "painting"]).ngroups),
                "emotion_counts": art["emotion"].value_counts().to_dict(),
                "pool_per_quadrant": pool["quadrant_artemis"].value_counts().reindex(QUADS).to_dict(),
                "min_annotations": MIN_ANNOTATIONS, "min_agreement": MIN_AGREEMENT})


In [ ]:
# Image download helpers 
import requests
from PIL import Image
HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) "
                         "Chrome/124.0 Safari/537.36"}

def _candidate_urls(painting):
    artist, title = painting.split("_", 1)
    urls = []
    try:   
        r = requests.get(f"https://www.wikiart.org/en/{artist}/{title}?json=2", headers=HEADERS, timeout=15)
        if r.ok:
            j = r.json()
            urls += [j[k] for k in ("image", "imageUrl") if isinstance(j.get(k), str)]
    except Exception:
        pass
    if not urls:
        try:  
            r = requests.get(f"https://www.wikiart.org/en/{artist}/{title}", headers=HEADERS, timeout=15)
            m = re.search(r'property="og:image"\s+content="([^"]+)"', r.text) if r.ok else None
            if m:
                urls.append(m.group(1))
        except Exception:
            pass
    urls += [f"https://uploads{i}.wikiart.org/images/{artist}/{title}.jpg" for i in range(9)]
    return urls

def download_image(painting, out_path):
    """Download one painting from WikiArt. Returns True on success."""
    if os.path.exists(out_path):
        return True
    for url in _candidate_urls(painting):
        url = url.split("!")[0]  
        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
            if not r.ok or len(r.content) < 5000:
                continue
            with open(out_path, "wb") as f:
                f.write(r.content)
            img = Image.open(out_path).convert("RGB")
            if min(img.size) < 64:
                raise ValueError("too small")
            img.save(out_path, "JPEG", quality=92)
            return True
        except Exception:
            if os.path.exists(out_path):
                os.remove(out_path)
    return False


In [ ]:
# Select 25 paintings per quadrant and download their images
SEL_CSV = P("results", "selected_paintings.csv")
if os.path.exists(SEL_CSV):
    sel = pd.read_csv(SEL_CSV)
    print("Using the earlier selection:", len(sel), "paintings")
else:
    rng = np.random.default_rng(SEED)
    rows, failures = [], {}
    for q in QUADS:
        cands = pool[pool.quadrant_artemis == q].sample(frac=1.0, random_state=int(rng.integers(1e9)))
        got, tried = 0, 0
        for _, r in cands.iterrows():
            if got == N_PER_QUADRANT or tried >= 20 * N_PER_QUADRANT:
                break
            tried += 1
            pid = f"P{len(rows) + 1:03d}"
            path = P("images", f"{pid}.jpg")
            if download_image(r.painting, path):
                rows.append({"pid": pid, **r.to_dict(), "image": path})
                got += 1
        failures[q] = tried - got
        print(f"{q}: {got} paintings ({tried - got} could not be downloaded)")
        if got == 0:
            raise RuntimeError("No images could be downloaded from WikiArt. Check the internet connection "
                               "and run this cell again.")
    sel = pd.DataFrame(rows)
    sel.to_csv(SEL_CSV, index=False)
    log("selection", {"per_quadrant": sel.quadrant_artemis.value_counts().reindex(QUADS).to_dict(),
                      "download_failures": failures,
                      "mean_agreement": sel.agreement.mean(),
                      "mean_annotations": sel.n_annotations.mean()})
print(sel.groupby("quadrant_artemis").agreement.describe()[["count", "mean", "min"]])


In [ ]:
# Colour–mood table of Sandve et al. (2016)
from sklearn.cluster import KMeans
from scipy import ndimage

COLOUR_TABLE = [
    ("blue",         ["0000FF", "00008B"],           {"calm": .8, "sad": .9}),
    ("dark blue",    ["403E5C"],                     {"calm": .4, "sad": .6, "sick": .1}),
    ("light blue",   ["ADD8E6"],                     {"calm": .3, "sad": .1}),
    ("yellow",       ["FFFF00", "E2B240"],           {"happy": 1.0, "excited": .4, "sick": .05}),
    ("light yellow", ["FFFFE0"],                     {"happy": .5, "excited": .2, "sick": .1}),
    ("orange",       ["FFA500", "FF8400"],           {"happy": .5, "excited": .7, "angry": .4}),
    ("red",          ["FF0000"],                     {"angry": .9, "excited": .8}),
    ("dark red",     ["8B0000"],                     {"angry": .9, "excited": .8}),
    ("green",        ["008000", "ADFF2F", "BBBD29"], {"calm": .6, "sick": .05, "happy": .1}),
    ("yellow-green", ["9ACD32"],                     {"sick": .8}),
    ("green-sick",   ["3E5C3F"],                     {"sick": .6}),
    ("black",        ["000000"],                     {"sad": .5, "angry": .6}),
    ("white",        ["FFFFFF"],                     {"happy": .2, "calm": .3}),
]
REF_RGB = np.array([[int(h[i:i + 2], 16) for i in (0, 2, 4)] for _, hexes, _ in COLOUR_TABLE for h in hexes], float)
REF_WEIGHTS = [w for _, hexes, w in COLOUR_TABLE for _ in hexes]

def colour_mood(path, k=15, max_side=256, min_component=0.001, seed=SEED):
    """k-means colour clustering -> connected components -> nearest reference colour -> summed mood weights."""
    img = Image.open(path).convert("RGB")
    img.thumbnail((max_side, max_side))
    arr = np.asarray(img, dtype=np.float32)
    h, w, _ = arr.shape
    km = KMeans(n_clusters=k, n_init=4, random_state=seed).fit(arr.reshape(-1, 3))
    labels = km.labels_.reshape(h, w)
    scores = dict.fromkeys(MOODS, 0.0)
    for c in range(k):
        components, n = ndimage.label(labels == c)
        if n == 0:
            continue
        sizes = np.bincount(components.ravel())[1:]
        weights = REF_WEIGHTS[int(np.argmin(((REF_RGB - km.cluster_centers_[c]) ** 2).sum(axis=1)))]
        size = sizes[sizes >= min_component * h * w].sum()  
        for mood, wt in weights.items():
            scores[mood] += size * wt
    return max(scores, key=scores.get), scores

if "mood_colour" not in sel.columns:
    moods = [colour_mood(p)[0] for p in sel.image]
    sel["mood_colour"] = moods
    sel["quadrant_colour"] = sel.mood_colour.map(MOOD_TO_Q)
    sel.to_csv(SEL_CSV, index=False)
print(sel.mood_colour.value_counts().to_string())


In [ ]:
# Variants A and B: rule-based composer of Sandve et al. (2016), reimplemented
MAJOR, MINOR = [0, 2, 4, 5, 7, 9, 11], [0, 2, 3, 5, 7, 8, 10]
PENTA = {"major": [0, 2, 4, 7, 9], "minor": [0, 3, 5, 7, 10]}

CHORD_PROBS = {
    "major": {"T": [(0, .33), (2, .34), (5, .33)], "P": [(3, .5), (1, .5)], "D": [(4, .7), (6, .3)]},
    "minor": {"T": [(0, .5), (2, .5)], "P": [(3, .45), (1, .10), (5, .45)], "D": [(4, .9), (6, .1)]},
}
MOOD_PARAMS = {"happy": ("major", 80), "angry": ("minor", 70), "sad": ("minor", 40), "calm": ("major", 40)}

def triad(key_root, mode, degree):
    scale = MAJOR if mode == "major" else MINOR
    return [key_root + scale[(degree + i) % 7] + 12 * ((degree + i) // 7) for i in (0, 2, 4)]

def choose(rng, options):
    degs, probs = zip(*options)
    return degs[rng.choice(len(degs), p=np.array(probs) / sum(probs))]

def melody_part(rng, key_root, mode, chords, refrain):
    """One measure (16 sixteenth-note slots) following the rules of Sandve et al. (Table 4.3), approximately."""
    penta = [key_root + 12 + p for p in PENTA[mode]] + [key_root + 24 + p for p in PENTA[mode]]
    notes, slot, rest_after_landing = [], 0, 0
    while slot < 16:
        chord = chords[slot // 4]
        chord_tones = [t + 12 for t in chord] + [t + 24 for t in chord]
        if slot == 0 and rng.random() < 0.95:                
            notes.append((slot, 2, chord[0] + 12)); slot += 2; continue
        if rest_after_landing > 0 and rng.random() < 0.5:  
            rest_after_landing -= 1; slot += 1; continue
        if rng.random() < 0.15:                             
            slot += 1; continue
        if slot % 4 == 0 and slot + 4 <= 16 and rng.random() < (0.45 if refrain else 0.2):
            notes.append((slot, 4, int(rng.choice(chord_tones)))); slot += 4; continue  
        if slot % 4 == 0 and rng.random() < 0.6:            
            notes.append((slot, 2, int(rng.choice(chord_tones)))); slot += 2; rest_after_landing = 3; continue
        dur = 1 if rng.random() < 0.5 else 2                 
        notes.append((slot, min(dur, 16 - slot), int(rng.choice(penta)))); slot += dur
    return notes

def compose(mood, seed, midi_path):
    rng = np.random.default_rng(seed)
    mode, bpm = MOOD_PARAMS[mood]
    key_root = 48 + int(rng.integers(12))                  
    chords = [triad(key_root, mode, choose(rng, CHORD_PROBS[mode][f])) for f in ("T", "P", "D", "D")]
    verse = melody_part(rng, key_root, mode, chords, refrain=False)
    refrain = melody_part(rng, key_root, mode, chords, refrain=True)
    structure = [("intro", None)] + [("verse", verse), ("refrain", refrain)] * 2
    sixteenth = 60.0 / bpm / 4
    pm = pretty_midi.PrettyMIDI(initial_tempo=bpm)
    piano = pretty_midi.Instrument(program=0, name="piano")
    t = 0.0
    while t < CLIP_SECONDS + 1:                              
        for _, melody in structure:
            for _ in range(int(rng.integers(1, 4))):          
                for i, chord in enumerate(chords):
                    start, end = t + 4 * i * sixteenth, t + 4 * (i + 1) * sixteenth
                    for pitch in chord:
                        piano.notes.append(pretty_midi.Note(55, pitch, start, end))
                    piano.notes.append(pretty_midi.Note(70, chord[0] - 12, start, end))  
                for s, d, pitch in (melody or []):
                    piano.notes.append(pretty_midi.Note(85, pitch, t + s * sixteenth, t + (s + d) * sixteenth))
                t += 16 * sixteenth
    pm.instruments.append(piano)
    pm.write(midi_path)
    return {"mode": mode, "bpm": bpm, "key_root": key_root}

# Variants A and B use the same seed for the same painting: when the colour table and ArtEmis
sel["seed"] = [SEED * 1000 + i for i in range(len(sel))]
sel["quadrant_intended_A"] = sel.quadrant_colour
sel["quadrant_intended_B"] = sel.quadrant_artemis
sel["quadrant_intended_C"] = sel.quadrant_artemis
sel.to_csv(SEL_CSV, index=False)
for variant, qcol in (("A", "quadrant_colour"), ("B", "quadrant_artemis")):
    for _, r in sel.iterrows():
        wav = P("audio", variant, f"{r.pid}.wav")
        if os.path.exists(wav):
            continue
        mid = P("midi", f"{variant}_{r.pid}.mid")
        compose(Q_TO_MOOD[r[qcol]], int(r.seed), mid)
        render_midi(mid, wav)
    print(f"Variant {variant}: {len(glob.glob(P('audio', variant, '*.wav')))} clips")
log("composer", {"mood_params": MOOD_PARAMS, "soundfont": os.path.basename(SOUNDFONT)})


In [ ]:
# MusicGen (small), one prompt per quadrant 
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration

PROMPTS = {
    "Q1": "A joyful, upbeat solo piano piece in a major key, fast tempo",
    "Q2": "A tense, agitated solo piano piece in a minor key, fast tempo, loud",
    "Q3": "A sad, melancholic solo piano piece in a minor key, slow tempo",
    "Q4": "A calm, peaceful solo piano piece in a major key, slow tempo",
}
todo = [r for _, r in sel.iterrows() if not os.path.exists(P("audio", "C", f"{r.pid}.wav"))]
if todo:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("WARNING: no GPU found. This step will be very slow. Use Runtime → Change runtime type → T4 GPU.")
    processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
    musicgen = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(device).eval()
    sr_mg = musicgen.config.audio_encoder.sampling_rate
    frame_rate = getattr(musicgen.config.audio_encoder, "frame_rate", 50)
    max_new_tokens = int(CLIP_SECONDS * frame_rate) + 10
    gen_cfg = {"do_sample": True, "guidance_scale": 3.0, "max_new_tokens": max_new_tokens,
               "top_k": int(getattr(musicgen.generation_config, "top_k", 250) or 250),
               "temperature": float(getattr(musicgen.generation_config, "temperature", 1.0) or 1.0)}
    t_start = time.time()
    for i, r in enumerate(todo):
        torch.manual_seed(int(r.seed))                 
        inputs = processor(text=[PROMPTS[r.quadrant_artemis]], padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            audio = musicgen.generate(**inputs, **gen_cfg)
        finish_clip(audio[0, 0].float().cpu().numpy(), sr_mg, P("audio", "C", f"{r.pid}.wav"))
        if (i + 1) % 10 == 0:
            print(f"{i + 1}/{len(todo)} clips, {(time.time() - t_start) / (i + 1):.1f} s per clip")
    log("musicgen", {"model": "facebook/musicgen-small", "prompts": PROMPTS, "generation": gen_cfg,
                     "sampling_rate": sr_mg, "device": device,
                     "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
                     "seconds_per_clip": (time.time() - t_start) / len(todo)})
    del musicgen
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print("Variant C:", len(glob.glob(P("audio", "C", "*.wav"))), "clips")


In [ ]:
# EMOPIA piano clips with known quadrants, used to check the automatic listener
import requests

def fetch_emopia_midis():
    """Download EMOPIA from Zenodo (or ask for an upload) and return a list of (midi_path, quadrant)."""
    raw = P("emopia", "raw")
    if not glob.glob(os.path.join(raw, "**", "*.mid"), recursive=True):
        zip_path = os.path.join(raw, "emopia.zip")
        for record in ("5090631", "5257995"):
            try:
                meta = requests.get(f"https://zenodo.org/api/records/{record}", timeout=30).json()
                entries = meta.get("files", [])
                if isinstance(entries, dict):
                    entries = list(entries.get("entries", entries).values())
                for f in entries:
                    key = f.get("key", "")
                    if key.lower().endswith(".zip") and "emopia" in key.lower():
                        url = f.get("links", {}).get("self") or f"https://zenodo.org/records/{record}/files/{key}?download=1"
                        print("Downloading", key, "from Zenodo record", record)
                        with requests.get(url, stream=True, timeout=120) as r:
                            r.raise_for_status()
                            with open(zip_path, "wb") as out:
                                for chunk in r.iter_content(1 << 20):
                                    out.write(chunk)
                        break
            except Exception as e:
                print("Zenodo record", record, "failed:", e)
            if os.path.exists(zip_path):
                break
        if not os.path.exists(zip_path):
            assert IN_COLAB, "Download EMOPIA manually and unzip it into " + raw
            print("Automatic download failed. Download the EMOPIA zip from https://zenodo.org/records/5090631 "
                  "and upload it here.")
            up = colab_files.upload()
            shutil.move(next(iter(up)), zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(raw)
    out = []
    for path in glob.glob(os.path.join(raw, "**", "*.mid*"), recursive=True):
        m = re.search(r"(?:^|[^A-Za-z0-9])(Q[1-4])_", os.path.basename(path)) or re.search(r"^(Q[1-4])_", os.path.basename(path))
        if m:
            out.append((path, m.group(1)))
    return out

EMO_CSV = P("results", "emopia_clips.csv")
if os.path.exists(EMO_CSV):
    emo = pd.read_csv(EMO_CSV)
else:
    midis = pd.DataFrame(fetch_emopia_midis(), columns=["midi", "quadrant"])
    print(f"EMOPIA: {len(midis)} MIDI clips found")
    print(midis.quadrant.value_counts().reindex(QUADS).to_string())
    parts = []
    for q in QUADS:
        sub = midis[midis.quadrant == q].sample(n=min(N_EMOPIA_PER_QUADRANT, (midis.quadrant == q).sum()),
                                                random_state=SEED)
        sub = sub.assign(split=["calibration" if i % 2 == 0 else "test" for i in range(len(sub))])
        parts.append(sub)
    emo = pd.concat(parts, ignore_index=True)
    emo["eid"] = [f"E{i + 1:03d}" for i in range(len(emo))]
    emo["wav"] = [P("emopia", "audio", f"{e}.wav") for e in emo.eid]
    emo.to_csv(EMO_CSV, index=False)
for _, r in emo.iterrows():
    if not os.path.exists(r.wav):
        render_midi(r.midi, r.wav)
log("emopia", {"n_clips": len(emo), "per_quadrant": emo.quadrant.value_counts().reindex(QUADS).to_dict(),
               "split": emo.split.value_counts().to_dict()})
print(len(emo), "EMOPIA clips rendered")


In [ ]:
# Automatic listener: Essentia valence/arousal model trained on DEAM
import requests
ESS = "https://essentia.upf.edu/models"
EMB_CANDIDATES = [f"{ESS}/feature-extractors/musicnn/msd-musicnn-1"]
HEAD_CANDIDATES = [f"{ESS}/classification-heads/deam/deam-msd-musicnn-2",
                   f"{ESS}/classification-heads/deam/deam-msd-musicnn-1",
                   f"{ESS}/classification-heads/emomusic/emomusic-msd-musicnn-2"]

def fetch_model(candidates):
    """Download the first available Essentia model (.pb graph + .json metadata)."""
    for base in candidates:
        name = base.rsplit("/", 1)[1]
        pb, js = P("models", name + ".pb"), P("models", name + ".json")
        try:
            for url, path in ((base + ".pb", pb), (base + ".json", js)):
                if not os.path.exists(path):
                    r = requests.get(url, timeout=120)
                    r.raise_for_status()
                    open(path, "wb").write(r.content)
            return pb, json.load(open(js))
        except Exception as e:
            print("Not available:", base, e)
            for path in (pb, js):
                if os.path.exists(path):
                    os.remove(path)
    raise RuntimeError("No Essentia model could be downloaded.")

def output_name(meta, purpose, default):
    for o in meta.get("schema", {}).get("outputs", []):
        if o.get("output_purpose") == purpose:
            return o["name"]
    return default

emb_pb, emb_meta = fetch_model(EMB_CANDIDATES)
head_pb, head_meta = fetch_model(HEAD_CANDIDATES)
classes = [c.lower() for c in head_meta.get("classes", ["valence", "arousal"])]
print("Listener:", head_meta.get("name"), "| trained on:", head_meta.get("dataset", {}).get("name"),
      "| outputs:", classes)

LISTENER_SCRIPT = P("models", "listener.py")
open(LISTENER_SCRIPT, "w").write('''
import sys, csv
import essentia.standard as es
emb_pb, emb_out, head_pb, head_out, list_file, out_csv = sys.argv[1:7]
emb = es.TensorflowPredictMusiCNN(graphFilename=emb_pb, output=emb_out)
head = es.TensorflowPredict2D(graphFilename=head_pb, output=head_out)
with open(out_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["path", "out0", "out1"])
    for path in open(list_file).read().split("\\n"):
        if path:
            audio = es.MonoLoader(filename=path, sampleRate=16000, resampleQuality=4)()
            pred = head(emb(audio)).mean(axis=0)     # average over the clip's time frames
            w.writerow([path, float(pred[0]), float(pred[1])])
''')

def run_listener(paths, out_csv):
    if not os.path.exists(out_csv):
        list_file = out_csv + ".txt"
        open(list_file, "w").write("\n".join(paths))
        r = subprocess.run([sys.executable, LISTENER_SCRIPT, emb_pb,
                            output_name(emb_meta, "embeddings", "model/dense/BiasAdd"), head_pb,
                            output_name(head_meta, "predictions", "model/Identity"), list_file, out_csv],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stderr[-3000:])
            raise RuntimeError("The listener failed (see the error above).")
    df = pd.read_csv(out_csv)
    df["valence"] = df["out0"] if classes[0].startswith("val") else df["out1"]
    df["arousal"] = df["out1"] if classes[0].startswith("val") else df["out0"]
    return df.set_index("path")[["valence", "arousal"]]

clip_paths = [P("audio", v, f"{pid}.wav") for v in VARIANTS for pid in sel.pid]
pred_clips = run_listener(clip_paths, P("results", "listener_clips.csv"))
pred_emo = run_listener(list(emo.wav), P("results", "listener_emopia.csv"))
log("listener", {"embedding_model": os.path.basename(emb_pb), "head_model": os.path.basename(head_pb),
                 "trained_on": head_meta.get("dataset", {}).get("name"), "classes": classes,
                 "description": head_meta.get("description")})
print("Predictions:", len(pred_clips), "clips,", len(pred_emo), "EMOPIA clips")


In [ ]:
# Check the listener on EMOPIA and calibrate its quadrant thresholds
emo_p = emo.join(pred_emo, on="wav")
cal, test = emo_p[emo_p.split == "calibration"], emo_p[emo_p.split == "test"]

THR = {"valence": float(cal.valence.median()), "arousal": float(cal.arousal.median())}
SD = {"valence": float(cal.valence.std()), "arousal": float(cal.arousal.std())}

def to_quadrants(df, thr):
    v, a = df.valence - thr["valence"], df.arousal - thr["arousal"]
    return [quadrant_of(x, y) for x, y in zip(v, a)]

def listener_check(df, thr):
    pq = pd.Series(to_quadrants(df, thr), index=df.index)
    true_v = df.quadrant.isin(["Q1", "Q4"])
    true_a = df.quadrant.isin(["Q1", "Q2"])
    return {"quadrant_accuracy": float((pq == df.quadrant).mean()),
            "valence_accuracy": float(((df.valence >= thr["valence"]) == true_v).mean()),
            "arousal_accuracy": float(((df.arousal >= thr["arousal"]) == true_a).mean()),
            "n": int(len(df)),
            "confusion": pd.crosstab(df.quadrant, pq).reindex(index=QUADS, columns=QUADS, fill_value=0).values.tolist()}

check_test = listener_check(test, THR)
check_mid = listener_check(test, {"valence": 5.0, "arousal": 5.0})
print(f"Listener on held-out EMOPIA clips (n={check_test['n']}): quadrant accuracy "
      f"{check_test['quadrant_accuracy']:.0%} (chance 25%), valence {check_test['valence_accuracy']:.0%}, "
      f"arousal {check_test['arousal_accuracy']:.0%} (chance 50%)")
print(f"Without calibration (midpoint 5): quadrant accuracy {check_mid['quadrant_accuracy']:.0%}")
log("listener_check", {"thresholds": THR, "sd": SD, "test": check_test, "test_midpoint_5": check_mid,
                       "prediction_range_clips": {k: [float(pred_clips[k].min()), float(pred_clips[k].max())]
                                                  for k in ("valence", "arousal")}})


In [ ]:
# Audio embeddings for the diversity measure 
import torch
from transformers import ClapModel, ClapProcessor
EMB_NPY = P("results", "clap_embeddings.npy")
if os.path.exists(EMB_NPY):
    EMB = np.load(EMB_NPY, allow_pickle=True).item()
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    clap = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(device).eval()
    cproc = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")

    def clap_embed(path):
        wav, _ = librosa.load(path, sr=48000, mono=True)
        half = 10 * 48000                         
        segs = [wav[:half], wav[half:2 * half]]
        try:
            inp = cproc(audios=segs, sampling_rate=48000, return_tensors="pt")
        except (TypeError, ValueError):
            inp = cproc(audio=segs, sampling_rate=48000, return_tensors="pt")
        inp = {k: v.to(device) for k, v in inp.items()}
        with torch.no_grad():
            e = clap.get_audio_features(**inp)
        if not torch.is_tensor(e):
            e = getattr(e, "pooler_output", None) if getattr(e, "pooler_output", None) is not None else e[0]
        e = torch.nn.functional.normalize(e, dim=-1).mean(0)
        return torch.nn.functional.normalize(e, dim=0).cpu().numpy()

    EMB = {(v, pid): clap_embed(P("audio", v, f"{pid}.wav")) for v in VARIANTS for pid in sel.pid}
    np.save(EMB_NPY, EMB, allow_pickle=True)
    log("clap", {"model": "laion/clap-htsat-unfused", "windows": "0-10 s and 10-20 s, averaged"})
print(len(EMB), "embeddings")


In [ ]:
# Run Experiments 1–4 and the statistics
from scipy import stats
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.contingency_tables import mcnemar

def prop_summary(successes, n, p0=0.25):
    """Rate, two-sided 95% Wilson interval, and one-sided binomial test against chance level p0."""
    p = stats.binomtest(int(successes), int(n), p0, alternative="greater").pvalue
    ci = stats.binomtest(int(successes), int(n), p0).proportion_ci(confidence_level=0.95, method="wilson")
    return {"k": int(successes), "n": int(n), "rate": successes / n, "ci95": [ci.low, ci.high], "p_vs_chance": p}

def crosstab(rows, cols):
    return pd.crosstab(pd.Series(list(rows), name="r"), pd.Series(list(cols), name="c")) \
             .reindex(index=QUADS, columns=QUADS, fill_value=0)

# Listener predictions for every clip
rows = []
for v in VARIANTS:
    for _, r in sel.iterrows():
        p = pred_clips.loc[P("audio", v, f"{r.pid}.wav")]
        zv = (p.valence - THR["valence"]) / SD["valence"]
        za = (p.arousal - THR["arousal"]) / SD["arousal"]
        rows.append({"variant": v, "pid": r.pid, "valence": p.valence, "arousal": p.arousal, "zv": zv, "za": za,
                     "predicted": quadrant_of(zv, za), "intended": r[f"quadrant_intended_{v}"],
                     "artemis": r.quadrant_artemis})
pred = pd.DataFrame(rows)
pred.to_csv(P("results", "predictions.csv"), index=False)
R = {}

# Experiment 1: colour table vs. ArtEmis
agree = (sel.quadrant_colour == sel.quadrant_artemis)
R["exp1"] = {**prop_summary(agree.sum(), len(sel)),
             "kappa": cohen_kappa_score(sel.quadrant_colour, sel.quadrant_artemis),
             "confusion_colour_rows_artemis_cols": crosstab(sel.quadrant_colour, sel.quadrant_artemis).values,
             "mood_counts": sel.mood_colour.value_counts().reindex(MOODS, fill_value=0).to_dict(),
             "colour_quadrant_counts": sel.quadrant_colour.value_counts().reindex(QUADS, fill_value=0).to_dict(),
             "agreement_per_artemis_quadrant": sel.assign(a=agree).groupby("quadrant_artemis").a.mean()
                                                 .reindex(QUADS).to_dict()}

# Experiment 2: emotional accuracy 
R["exp2"] = {}
for v in VARIANTS:
    d = pred[pred.variant == v]
    hit_i, hit_a = d.predicted == d.intended, d.predicted == d.artemis
    pos_v = d.intended.isin(["Q1", "Q4"]); pos_a = d.intended.isin(["Q1", "Q2"])
    R["exp2"][v] = {"vs_intended": prop_summary(hit_i.sum(), len(d)),
                    "vs_artemis": prop_summary(hit_a.sum(), len(d)),
                    "valence_accuracy": prop_summary(((d.zv >= 0) == pos_v).sum(), len(d), 0.5),
                    "arousal_accuracy": prop_summary(((d.za >= 0) == pos_a).sum(), len(d), 0.5),
                    "per_quadrant": d.assign(h=hit_i).groupby("intended").h.mean().reindex(QUADS).to_dict(),
                    "per_quadrant_n": d.intended.value_counts().reindex(QUADS, fill_value=0).to_dict(),
                    "confusion_intended_rows_predicted_cols": crosstab(d.intended, d.predicted).values,
                    "predicted_counts": d.predicted.value_counts().reindex(QUADS, fill_value=0).to_dict()}

def mcnemar_test(x, y):
    t = pd.crosstab(pd.Categorical(x, [False, True]), pd.Categorical(y, [False, True]), dropna=False).values
    res = mcnemar(t, exact=True)
    return {"table_[x_wrong,x_right]x[y_wrong,y_right]": t, "p": res.pvalue}

hits = {v: (pred[pred.variant == v].set_index("pid").predicted ==
            pred[pred.variant == v].set_index("pid").artemis) for v in VARIANTS}
R["exp2"]["A_vs_B_vs_artemis"] = mcnemar_test(hits["A"].values, hits["B"].values)
R["exp2"]["B_vs_C_vs_intended"] = mcnemar_test(hits["B"].values, hits["C"].values)
R["exp2"]["identical_A_B_clips"] = int((sel.quadrant_colour == sel.quadrant_artemis).sum())

# Experiment 3: matching 
rng = np.random.default_rng(SEED + 3)
pids = list(sel.pid)
paint_pt = sel.set_index("pid")[["painting_v", "painting_a"]]
distractor_sets = {pid: [list(rng.choice([p for p in pids if p != pid], 3, replace=False))
                         for _ in range(N_DISTRACTOR_SETS)] for pid in pids}   # same sets for all variants
match = {}
for v in VARIANTS:
    d = pred[pred.variant == v].set_index("pid")
    per_painting = []
    for pid in pids:
        clip = np.array([d.loc[pid, "zv"], d.loc[pid, "za"]])
        wins = []
        for k, dis in enumerate(distractor_sets[pid]):
            cands = [pid] + dis
            dist = np.array([np.linalg.norm(clip - paint_pt.loc[c].values) for c in cands])
            best = np.flatnonzero(np.isclose(dist, dist.min()))
            wins.append(1.0 / len(best) if 0 in best else 0.0)    # ties are shared equally
        per_painting.append(np.mean(wins))
    match[v] = np.array(per_painting)
R["exp3"] = {}
for v in VARIANTS:
    x = match[v]
    boots = [rng.choice(x, len(x)).mean() for _ in range(5000)]
    R["exp3"][v] = {"rate": x.mean(), "ci95": [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))],
                    "p_vs_chance": stats.wilcoxon(x - 0.25, alternative="greater").pvalue if np.any(x != 0.25) else 1.0,
                    "per_quadrant": pd.Series(x, index=pids).groupby(sel.set_index("pid").quadrant_artemis).mean()
                                     .reindex(QUADS).to_dict()}
R["exp3"]["A_vs_B_p"] = stats.wilcoxon(match["A"], match["B"]).pvalue if np.any(match["A"] != match["B"]) else 1.0
R["exp3"]["B_vs_C_p"] = stats.wilcoxon(match["B"], match["C"]).pvalue if np.any(match["B"] != match["C"]) else 1.0
pd.DataFrame(match, index=pids).to_csv(P("results", "matching_per_painting.csv"))

# Experiment 4: diversity within a quadrant 
def pairwise_distances(vecs):
    m = np.stack(vecs)
    return 1 - m @ m.T

def mean_distance(D, idx):
    """Mean distance over pairs of the (resampled) clips, skipping pairs of a clip with itself."""
    i, j = np.triu_indices(len(idx), 1)
    keep = idx[i] != idx[j]
    return float(D[idx[i][keep], idx[j][keep]].mean()) if keep.any() else float("nan")

R["exp4"] = {}
for v in VARIANTS:
    d = pred[pred.variant == v]
    mats = {q: pairwise_distances([EMB[(v, pid)] for pid in d[d.intended == q].pid])
            for q in QUADS if (d.intended == q).sum() > 1}
    per_q = {q: mean_distance(D, np.arange(len(D))) for q, D in mats.items()}
    boot = [np.nanmean([mean_distance(D, rng.integers(0, len(D), len(D))) for D in mats.values()])
            for _ in range(1000)]
    intended = dict(zip(d.pid, d.intended))
    allD = pairwise_distances([EMB[(v, pid)] for pid in d.pid])
    pid_list = list(d.pid)
    between = np.mean([allD[i, j] for i in range(len(pid_list)) for j in range(i + 1, len(pid_list))
                       if intended[pid_list[i]] != intended[pid_list[j]]])
    R["exp4"][v] = {"within_quadrant": float(np.nanmean(list(per_q.values()))),
                    "ci95": [float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))],
                    "per_quadrant": {q: per_q.get(q, float("nan")) for q in QUADS},
                    "between_quadrants": float(between)}

log("results", R)
for v in VARIANTS:
    print(f"{VARIANT_NAMES[v]:<26} emotional acc. {R['exp2'][v]['vs_intended']['rate']:.0%} | "
          f"vs ArtEmis {R['exp2'][v]['vs_artemis']['rate']:.0%} | matching {R['exp3'][v]['rate']:.0%} | "
          f"diversity {R['exp4'][v]['within_quadrant']:.3f}")
print(f"Exp. 1: colour table agrees with ArtEmis for {R['exp1']['rate']:.0%} of paintings (kappa {R['exp1']['kappa']:.2f})")


In [ ]:
# Figures for the report
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 10,
                     "axes.edgecolor": "#52514e", "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e",
                     "ytick.color": "#52514e", "axes.spines.top": False, "axes.spines.right": False,
                     "axes.linewidth": 0.8, "savefig.bbox": "tight", "savefig.pad_inches": 0.03})
COLORS = {"A": "#2a78d6", "B": "#eb6834", "C": "#1baf7a"}   # validated categorical slots 1-3
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e5e4e0"
FIG = lambda name: P("figures", name)

def heatmap(ax, M, xlabel, ylabel, title=None):
    M = np.asarray(M)
    ax.imshow(M, cmap="Blues", vmin=0, vmax=max(M.max(), 1))
    for i in range(4):
        for j in range(4):
            ax.text(j, i, int(M[i, j]), ha="center", va="center", fontsize=9,
                    color="white" if M[i, j] > 0.55 * M.max() else INK)
    ax.set_xticks(range(4)); ax.set_xticklabels(QUADS); ax.set_yticks(range(4)); ax.set_yticklabels(QUADS)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    if title:
        ax.set_title(title, fontsize=10, color=INK)

# Experiment 1: colour table vs. ArtEmis
fig, ax = plt.subplots(figsize=(3.0, 2.8))
heatmap(ax, R["exp1"]["confusion_colour_rows_artemis_cols"], "Quadrant from ArtEmis annotations",
        "Quadrant from colour table")
fig.savefig(FIG("fig_exp1_confusion.pdf")); plt.close(fig)

# Experiments 2 and 3: accuracy per variant with 95% confidence intervals
fig, axes = plt.subplots(1, 2, figsize=(6.3, 2.5), sharey=True)
panels = [("Emotional accuracy", {v: R["exp2"][v]["vs_intended"] for v in VARIANTS}),
          ("Matching accuracy", {v: R["exp3"][v] for v in VARIANTS})]
for ax, (title, res) in zip(axes, panels):
    for i, v in enumerate(VARIANTS):
        rate, (lo, hi) = res[v]["rate"], res[v]["ci95"]
        ax.bar(i, rate * 100, width=0.6, color=COLORS[v], edgecolor="white", linewidth=1.5, zorder=2)
        ax.errorbar(i, rate * 100, yerr=[[100 * (rate - lo)], [100 * (hi - rate)]], color=INK2,
                    capsize=3, linewidth=1, zorder=3)
        ax.text(i, hi * 100 + 2, f"{rate:.0%}", ha="center", va="bottom", fontsize=9, color=INK)
    ax.axhline(25, color=INK2, linestyle=(0, (4, 3)), linewidth=1, zorder=1)
    ax.text(2.42, 25, "chance", ha="left", va="center", fontsize=8, color=INK2,
            bbox=dict(facecolor="white", edgecolor="none", pad=1))
    ax.set_xticks(range(3)); ax.set_xticklabels(["A", "B", "C"]); ax.set_xlim(-0.5, 2.95)
    ax.set_title(title, fontsize=10, color=INK, pad=8)
    ax.set_ylim(0, 112); ax.set_yticks(range(0, 101, 20)); ax.yaxis.grid(True, color=GRID, linewidth=0.8, zorder=0); ax.set_axisbelow(True)
axes[0].set_ylabel("Accuracy (%)")
fig.savefig(FIG("fig_accuracy.pdf")); plt.close(fig)

# Experiment 2: intended vs. predicted quadrant per variant
fig, axes = plt.subplots(1, 3, figsize=(6.3, 2.3))
for ax, v in zip(axes, VARIANTS):
    heatmap(ax, R["exp2"][v]["confusion_intended_rows_predicted_cols"], "Predicted",
            "Intended" if v == "A" else "", VARIANT_NAMES[v])
fig.tight_layout(); fig.savefig(FIG("fig_confusion_variants.pdf")); plt.close(fig)

# Experiment 4: diversity within each quadrant
fig, ax = plt.subplots(figsize=(4.6, 2.5))
w = 0.26
for i, v in enumerate(VARIANTS):
    vals = [R["exp4"][v]["per_quadrant"][q] for q in QUADS]
    ax.bar(np.arange(4) + (i - 1) * w, vals, width=w, color=COLORS[v], edgecolor="white", linewidth=1.5,
           label=VARIANT_NAMES[v], zorder=2)
ax.set_xticks(range(4)); ax.set_xticklabels(QUADS)
ax.set_ylabel("Mean cosine distance")
ax.yaxis.grid(True, color=GRID, linewidth=0.8, zorder=0); ax.set_axisbelow(True)
ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.0, 1.0))
fig.savefig(FIG("fig_diversity.pdf")); plt.close(fig)

# Listener check on held-out EMOPIA clips
fig, ax = plt.subplots(figsize=(3.0, 2.8))
heatmap(ax, LOG["listener_check"]["test"]["confusion"], "Predicted by the listener", "EMOPIA label")
fig.savefig(FIG("fig_listener_emopia.pdf")); plt.close(fig)
print("Figures:", sorted(os.listdir(P("figures"))))


In [ ]:
# Package the results and download `results.zip`
import torch, transformers, sklearn, scipy
log("environment", {"python": platform.python_version(), "torch": torch.__version__,
                    "transformers": transformers.__version__, "sklearn": sklearn.__version__,
                    "scipy": scipy.__version__,
                    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                    "runtime_minutes_this_session": (time.time() - T0) / 60})

examples = []
for v in VARIANTS:
    for q in QUADS:
        pids = sel[sel[f"quadrant_intended_{v}"] == q].pid.head(2)
        examples += [P("audio", v, f"{pid}.wav") for pid in pids]
zip_path = "/content/results.zip" if IN_COLAB else P("results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(P("results", "*")):
        if not f.endswith(".txt"):
            z.write(f, os.path.join("results", os.path.basename(f)))
    for f in glob.glob(P("figures", "*")):
        z.write(f, os.path.join("figures", os.path.basename(f)))
    for f in examples:
        z.write(f, os.path.join("example_audio", os.path.basename(os.path.dirname(f)) + "_" + os.path.basename(f)))
print(f"results.zip: {os.path.getsize(zip_path) / 1e6:.1f} MB")
if IN_COLAB:
    colab_files.download(zip_path)
